# Training a PyTorch-style MLP **in the browser** (WebAssembly)

This notebook runs entirely in a **WebAssembly `xeus-python` kernel** (from the
[emscripten-forge](https://emscripten-forge.org) channel) inside JupyterLite — no
server, no local Python. It fits a small multi-layer perceptron (MLP) on a
synthetic dataset and shows the training loss decreasing over epochs.

> **What is `microtorch`?** Upstream PyTorch does not build for `wasm32-emscripten`,
> so this demo uses **`microtorch`**: a tiny, pure-Python, numpy-backed reverse-mode
> autograd engine that reimplements a *small PyTorch-compatible API subset*
> (`tensor`, `.backward()`, `nn.Linear/ReLU/Sequential/MSELoss`, `optim.SGD`).
> **It is not upstream PyTorch.** Its gradients are validated numerically against
> real PyTorch on the host (see the repo's `RESULTS.md`). For *real* PyTorch
> operators compiled to WASM (inference only), see the ExecuTorch WASM path.


In [ ]:
import numpy as np
import microtorch as torch          # reduced, PyTorch-compatible API
from microtorch import nn, optim
print('microtorch', torch.__version__, '| numpy', np.__version__)
import platform; print('running on', platform.machine(), 'python', platform.python_version())

## 1. Synthetic dataset
A nonlinear target: `y = relu(X @ w_true) + small noise`, so a linear model cannot
fit it well but a 1-hidden-layer MLP can.

In [ ]:
torch.manual_seed(0)
rng = np.random.RandomState(1)
N, D = 256, 4
X_np = rng.randn(N, D).astype(np.float32)
w_true = np.array([[1.5], [-2.0], [0.5], [3.0]], dtype=np.float32)
y_np = np.maximum(X_np @ w_true, 0.0) + 0.1 * rng.randn(N, 1).astype(np.float32)
X, y = torch.tensor(X_np), torch.tensor(y_np)
print('X', X.shape, '| y', y.shape)

## 2. Define the MLP, loss and optimizer
Idiomatic PyTorch-style code — `nn.Sequential`, `nn.Linear`, `nn.ReLU`,
`nn.MSELoss`, `optim.SGD`.

In [ ]:
model = nn.Sequential(
    nn.Linear(D, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
)
loss_fn = nn.MSELoss()
opt = optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
n_params = sum(int(np.prod(p.shape)) for p in model.parameters())
print('trainable tensors:', len(model.parameters()), '| scalar params:', n_params)

## 3. Training loop
Full-batch gradient descent for 200 epochs; the loss should fall by ~2-3 orders
of magnitude.

In [ ]:
losses = []
for epoch in range(200):
    opt.zero_grad()
    pred = model(X)
    loss = loss_fn(pred, y)
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if epoch % 25 == 0 or epoch == 199:
        print(f'epoch {epoch:3d}  loss = {loss.item():.5f}')
print(f'\nfinal loss {losses[-1]:.5f}  (started at {losses[0]:.5f}, '
      f'{losses[0]/losses[-1]:.0f}x reduction)')
assert losses[-1] < 0.1 * losses[0], 'loss did not decrease enough'

## 4. Plot the loss curve

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 4))
plt.semilogy(losses)
plt.xlabel('epoch'); plt.ylabel('MSE loss (log scale)')
plt.title('MLP training loss in WebAssembly (microtorch)')
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Sanity check: predictions track targets
Correlation between predictions and targets should be high (~0.99).

In [ ]:
with torch.no_grad():
    p = model(X).numpy().ravel()
t = y.numpy().ravel()
corr = float(np.corrcoef(p, t)[0, 1])
print(f'pred/target correlation: {corr:.4f}')
assert corr > 0.95